# 神经网络的学习

在上述学习中，我们运用了三个函数实现了三层神经网络，但是其中的init_network函数(即决定权重参数和偏置的函数)是调用的已经学习好的参数，并没有实现“学习”这一目标，只能算得上是套用，这一节将要学习，如何通过深度学习来自动确定最佳参数

背景 ：以手写数字识别为例，人类能在看到数字的时候以习惯，直觉认出那是什么数字，但显然机器没有这种直觉，于是传统的机器学习将图像转换为人为确定的特征量（SIFT,HOC等)，通过机器学习中SVM,KNN等方法，识别特征量来做到预测，但是这也有局限性，因为特征量是人为确定的，对于不同的问题如手写数字识别和认出猫狗这种问题，需要人来重新确立特征量，这很不好，于是现在有了深度学习，整个过程不需要人参与，通过神经网络直接处理数据并给出答案

### 前置的知识：

### 1. 损失函数

在学习过程中，需要指标来确定当前学习的程度，损失函数就是这个指标，它可以是任意函数，常见的是均方误差和交叉熵误差

- 均方误差  $$
\mathcal{L}_{\text{MSE}} = \frac{1}{2} \sum_{i=1}^{N} (y_i - t_k)^2
$$

$y_i$表示的是神经网络输出结果，$\hat{y}_i$则表示测试集数据,$t_k%$表示监督数据(验证集)，将正确解标签设为1，其余设为零，这种方法叫做one-hot表示

In [2]:
import numpy as np

def mean_square_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

# 设2为正解
t = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0])

# 例1
y1 = np.array([0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0])
print("例1误差:", mean_square_error(y1, t))

# 例2（补全了10个值）
y2 = np.array([0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.6, 0.0, 0.0, 0.0])
print("例2误差:", mean_square_error(y2, t))

例1误差: 0.09750000000000003
例2误差: 0.5975


- 交叉熵误差

$$
\mathcal{L} = -\sum_{k=1}^{K} t_k \log y_k
$$

同样的$t_k$表示的是one_hot方法下的标签集，所以事实上，这个误差只计算了对应正确解标签的输出的自然对数，也就是说$y$本来是一个有着十个元素的一维数组，假设正确标签为1，而y=[0.1,0.2,0.3,0.4,0.0,0.0,0.0,0.0,0.0,0.0]，则只有0.2参与到计算之中，当这个概率越大时，$logy_k$自然越大，但是由于取了相反数的原因会变小，实现了预测成功时误差更小，下面是代码实现

In [3]:
import numpy as np

def cross_entropy_error(y, t):
    delta = 1e-7#这里的delta是一个保护作用，当概率出现log(0)时会变成负无穷，导致无法计算
    return -np.sum(t * np.log(y + delta))

# 设2为正解
t = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0])

# 例1
y = np.array([0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0])
print("例1交叉熵误差:", cross_entropy_error(y, t))

# 例2
y = np.array([0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.6, 0.0, 0.0, 0.0])
print("例2交叉熵误差:", cross_entropy_error(y, t))

例1交叉熵误差: 0.510825457099338
例2交叉熵误差: 2.302584092994546


### 2. mini—batch学习

前面讲的损失函数，都是针对一个训练数据的(如一张图片)，但是要评价一个模型需要很多数据，这时候就要求所有训练数据的损失函数的总和：
$$
L = -\frac{1}{N} \sum_{n=1}^{N} \sum_{k=1}^{K} t_{nk} \log y_{nk}
$$

实际上就是把每一个加起来求平均，这还是为了正规化

但是往往整个数据集有较大数据，所有都测试一遍太费时间，所以我们选出一部分来测，如从MNIST的60000个中选100个，这就是mini-batch学习

In [9]:
import numpy as np
from mnist import load_mnist
(x_train, t_train), (x_test, t_test) = load_mnist(flatten=True, normalize=True, one_hot_label=True)#不加one_hot时t就是60000个元素的一维矩阵，每个元素表示正确答案，而加了之后就是用one_hot方法来表示正确答案(即一个10个元素的数组，只有正确位置的元素为1，其余都为零)
print(x_train.shape, t_train.shape)#以上是调入MNIST库的方法
train_size=x_train.shape[0]#右式的结果是60000，就是x_train的行数
batch_size=10
batch_mask=np.random.choice(train_size, batch_size)#这个的作用相当于从0到train_size任取batch_size个数
x_batch=x_train[batch_mask]
t_batch=t_train[batch_mask]
print(batch_mask)



(60000, 784) (60000, 10)
[34697 20561 13953 40966 23236 17114 15996 26956 25519  1989]


下面正式实现mini-batch版的交叉熵误差

In [10]:
import numpy as np
def cross_entropy_error(y,t):
    if y.ndim==1:
        t=t.reshape(1,t.size)
        y=y.reshape(1,y.size)
    batch_size=y.shape[0]#获取一批次里面的数据个数
    return -np.sum(t*np.log(y+0.000001))/batch_size

以上代码的核心逻辑与处理单个数据时相同，这是在one-hot方法下的计算方法，这时我们可以视作是学过的批处理，当处理的数据不止一个时，传入的y，t实际上变成了二维矩阵(单个数据时只是十个元素的一维矩阵)，多的一维就是数据的个数，所以上述的if语句是为了处理刚好只输入一个数据时的情况

当然，t也可以不用one-hot方法表示，直接用标签形式，只用稍加改动代码

In [12]:
import numpy as np
def cross_entropy_error(y,t):
    if y.ndim==1:
        t=t.reshape(1,t.size)
        y=y.reshape(1,y.size)
    batch_size=y.shape[0]#获取一批次里面的数据个数
    return -np.sum(np.log(y[np.arange(batch_size),t]))/batch_size
#上面这一行非常简单，由于使用one-hot方法时，我们根本用不上除正确标签外的其他数据，所以在不使用one-hot时我们同样要这样做，y[np.arange(batch_size),t]这一句非常巧妙，利用了python的化石索引，首先生成从零到batch-size-1的一个数组称为o，对应y里面的batch-size张图片。然后和同样是一维数组的t对应结合，o的第i个和t的第i个结合，就能找到第i张图片输出层所给出的概率值

### 损失函数的意义

可能会有疑问为什么不直接使用识别精度来判断模型好坏的指标，这是因为我们要做的时不断更新参数，如果使用识别精度作为指标时，会使绝大部分地方的导数变为零从而不能更新参数，归根结底，识别精度不会因为参数的细微改变发生变化，即使有变化，也是突兀的，不连续的变化，不适合作为更新参数的指标。同理我们不使用阶跃函数作为神经网络的输出函数，因为它不连续，甚至没有导数，而连续且可导的sigmoid函数则非常合适

### 3.数值微分

既然了解到了求导可以推进参数的更新，那我们必然要学习如何在计算机上求导

$$
f'(x) = \lim_{\Delta x \to 0} \frac{f(x + \Delta x) - f(x)}{\Delta x}
$$

这是我们常见的求导方法，基于这个方法我们可以这么写：

In [ ]:
def numerical_diff(f,x):
    Δx=10e-50
    return (f(x+Δx) - f(x))/Δx

我们尽力给Δ取很小的值，但是这始终不是真正的导数定义，因为其产生了舍入误差，那怎么才能尽量减少误差的求导，需要使用中心差分法

$$
f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}
$$
利用这个我们来试试

In [ ]:
def function(x):
    return 0.01*x**2+0.1*x
def numerical_diff(f,x):
    h=1e-4
    return (f(x+h) - f(x-h))/(2*h)
import numpy as np
import matplotlib.pyplot as plt  # 加上 .pyplot
x=np.arange(0.0,20,0.1)
y=function(x)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.plot(x,y)
plt.show()
numerical_diff(function,5)
numerical_diff(function,10)
print(numerical_diff(function,5))
print(numerical_diff(function,10))#在一定精度范围内，已经非常接近了

### 4.梯度

梯度是一个向量，由全部变量的偏导数组成$$
\nabla f = \begin{pmatrix} \frac{\partial f}{\partial x_0}, \frac{\partial f}{\partial x_1} \end{pmatrix}
$$

In [ ]:
#梯度的实现
def numerical_gradient(f,x):
    h=1e-4
    grad=np.zeros_like(x)#生成一个形状与x相同，但所有元素为零的一个数组
    for idx in range(x.size):
        tmp=x[idx]
        x[idx]=tmp+h
        fxh1=f(x)
        x[idx]=tmp-h
        fxh2=f(x)
        grad[idx]=(fxh1-fxh2)/(2*h)
        x[idx]=tmp
    return grad
    #看似复杂，实则就是通过for结构计算每一个未知数的偏导数，然后合并到grad数组中


- 梯度的作用

经过高等数学的学习，我们知道，梯度指向的方向是各点函数值减小最多的方向(这里的方向是负梯度向量的方向)，非常有用！是梯度法中变量的更新方向

- 梯度法

梯度的方向指向梯度趋于零的地方，但是这不一定是取得最值的地方，有可能只是一个鞍点(即极值点)，但是它能够最大限度的减少函数的值，所以我们要利用梯度，核心思想是：函数的取值从当前位置沿梯度方向前进一定距离，然后重新求梯度，再前进，这样的方法就是梯度法，其中寻找最小值的方法叫梯度下降法，相反寻找最大值的方法叫梯度上升法，但本质上并无区别

In [21]:
#梯度下降法的实现
def gradient_decent(f,lr=0.01,step_num=100):
    x=init_x#initx_x是初始值
    for i in range(step_num):
        grad=numerical_gradient(f,x)
        x-=lr*grad
    return x

利用上述代码可以求多元函数最小值如：f(x0+x1)=x0²+x1²

In [24]:
import numpy as np

# 1. 定义目标函数
def fun(x):
    return x[0]**2 + x[1]**2

# 2. 定义数值梯度计算函数（中心差分法的向量升级版）
def numerical_gradient(f, x):
    h = 1e-4  # 极小的步长
    grad = np.zeros_like(x)  # 生成和x形状一样的全0数组，用来存梯度

    for idx in range(x.size):
        tmp_val = x[idx]

        # 计算 f(x+h)
        x[idx] = tmp_val + h
        fxh1 = f(x)

        # 计算 f(x-h)
        x[idx] = tmp_val - h
        fxh2 = f(x)

        # 中心差分公式
        grad[idx] = (fxh1 - fxh2) / (2 * h)
        x[idx] = tmp_val  # 还原当前维度的值，不影响下一次循环

    return grad

# 3. 定义梯度下降函数（把初始值作为参数传进来）
def gradient_decent(f, init_x, lr=0.01, step_num=100):
    x = init_x
    for i in range(step_num):
        grad = numerical_gradient(f, x)
        x -= lr * grad
    return x

# 4. 执行并打印结果
init_x = np.array([-3.0, 4.0])
final_x = gradient_decent(fun, init_x, lr=0.1, step_num=100)

print("最终收敛的坐标 x:", final_x)
print("该点的函数值 fun(x):", fun(final_x))

最终收敛的坐标 x: [-6.11110793e-10  8.14814391e-10]
该点的函数值 fun(x): 1.0373788922158197e-18


上述中的lr表示学习率，不能太大也不能太小，要取合适的，这样的参数称为超参数，不同于权重和偏置由机器训练得到，超参数是由人工设定的，一般要尝试多次

- 神经网络的梯度

神经网络也有梯度，这个梯度指的是损失函数关于权重参数的梯度(用来寻找参数迭代的方向)。

我们知道权重参数w也可以是矩阵组成，那么梯度就是要对每一个矩阵元素求偏导，下面是简单的求神经网络的梯度,但是首先要定义一个名为simpleNet的类

In [7]:
import numpy as np
from common.functions import softmax, cross_entropy_error#借用一下common包里面的softmax，cross_entropy_error函数
from common.gradient import numerical_gradient#借用梯度求导


class simpleNet:
    def __init__(self):
        self.W = np.random.randn(2,3)

    def predict(self, x):
        return np.dot(x, self.W)#点乘，推进神经网络

    def loss(self, x, t):
        z = self.predict(x)#得到wx的结果，注意这里并没有加上偏置
        y = softmax(z)#对每一行处理，(因为每一行代表一张图片)每一行加起来的概率为1
        loss = cross_entropy_error(y, t)

        return loss
#我们可以简单的将类理解为一个包含了变量和变量处理方法的类结构体，在上述类中，我们定义了变量self.W并用高斯分布使其初始化为2*3的矩阵，然后定义了predict，loss函数
net=simpleNet()
print(net.W)
x=np.array([0.6,0.9])
p=net.predict(x)
print(p)
print(np.argmax(p))
t=np.array([0,0,1])
#计算的是初始参数的情况下的损失函数，还没有开始学习，接下来求梯度
def f(w):
    return net.loss(x,t)#只是个形式而已，套个壳，因为求导函数需要的是一个函数的形式而不是方法
dw=numerical_gradient(f,net.W)
print(dw)
#这一段就是对w权重参数求一手梯度，梯度是一个与w形状相同的矩阵，我们可以清晰的通过每一个元素的数值看到其稍微改变对损失函数的影响，下面利用梯度法更新权重参数就可以实现学习

[[-0.27033508  2.00757949  1.17670616]
 [ 2.21382911  0.91614792 -0.66732077]]
[1.83024515 2.02908083 0.105435  ]
1
[[ 0.25018852  0.30522536 -0.55541388]
 [ 0.37528278  0.45783804 -0.83312082]]


### 5.学习算法的实现！！！

前置知识已经学完了，下面要真正实现学习算法，来梳理一下步骤
1. mini-batch：由于数据一般很大，全部训练时间太长，我们需要挑选部分数据来进行训练
2. 计算梯度：为了减少mini—batch的损失函数的值，需要求出各个权重参数的梯度
3. 更新参数：将权重参数沿着梯度方向更新
4. 重复步骤1，2，3直到损失函数足够小

上面这个方法，由于mini-batch是随机挑选的，所以叫做随机梯度下降法，简称是SGD，下面我们来写出第一个两层神经网络！

In [ ]:

from common.functions import *
from common.gradient import numerical_gradient
import numpy as np


class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
       #初始化权重，inputsize，hidden_size分别是输入层和隐藏层的大小
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)#“weight_init_std *”,将生成的随机数乘以0.01，避免随机生成的权重参数太大
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']

        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)

        return y#这一段是输入层到隐藏层，隐藏层到输出层，因为是两层神经网络，所以进行了两次线性运算，而sigmoid函数是为了避免两次线性预算丧失隐藏层的作用


    def loss(self, x, t):
        y = self.predict(x)

        return cross_entropy_error(y, t)#损失函数

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)#axis等于一告诉我们是去一行里面最大的那一个即结果
        t = np.argmax(t, axis=1)#同理t也是这样

        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy#计算准确度


    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)#这个lambda是上面f(x)形式的简化不重要

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads#计算的是每一个参数的梯度并装在名为grads的字典里，方便学习

    def gradient(self, x, t):#灵魂步骤，涉及误差反向传播法，目前还没学
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        grads = {}

        batch_num = x.shape[0]

        # forward
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)

        # backward
        dy = (y - t) / batch_num
        grads['W2'] = np.dot(z1.T, dy)
        grads['b2'] = np.sum(dy, axis=0)

        dz1 = np.dot(dy, W2.T)
        da1 = sigmoid_grad(a1) * dz1
        grads['W1'] = np.dot(x.T, da1)
        grads['b1'] = np.sum(da1, axis=0)

        return grads
#到这里我们实现了大致的流程，但是缺少了mini—batch的实现，下面补上

In [ ]:

import sys, os
sys.path.append(os.pardir)
import numpy as np
import matplotlib.pyplot as plt
from mnist import load_mnist
from two_layer_net import TwoLayerNet


(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)
#下面是一些超参数
iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]
    #获取mini-batch

    grad = network.gradient(x_batch, t_batch)
    #计算梯度，这个是高速版，利用的是误差反向传播法

    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)
    #这一步是在更新参数了
    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print("train acc, test acc | " + str(train_acc) + ", " + str(test_acc))


markers = {'train': 'o', 'test': 's'}
x = np.arange(len(train_acc_list))
plt.plot(x, train_acc_list, label='train acc')
plt.plot(x, test_acc_list, label='test acc', linestyle='--')
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.ylim(0, 1.0)
plt.legend(loc='lower right')
plt.show()#后面是利用mat使训练过程可视化

看到上面的代码，可能还会有些疑惑，train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)，这到底是在干嘛，其实回归到神经网络的目的：掌握泛化能力，也就意味着要防止过拟合现象出现，所以要定期对训练数据和测试数据记录记录识别精度，而一个周期就是一个epoch，一个epoch表示学习中所有训练数据均被使用过一次时的更新次数，如果10000个数据，一个mini-batch抽10个，那要1000次抽完，那么一个epoch就是1000.当经经历一个epoch时再记录，能避免过多的记录导致浪费太多时间，通过上图也可以发现，随着epoch的前进，识别精度也在提高了，且训练集和测试集的两条线基本重合，说明也没有出现过拟合